# Import Geopolitical Risk Index (GPR)
- Author: Bryan Bravo
- Created: 2026-03-23
## Import Libraries

In [1]:
import os
import sys

# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session

In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


### Custom Functions

### Variables

## Query

In [ ]:
raw_df = pd.read_excel("https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls",
                       header=0,
                       engine='xlrd')



In [4]:
raw_df

,month,GPR,GPRT,GPRA,GPRH,GPRHT,GPRHA,SHARE_GPR,N10,SHARE_GPRH,...,GPRHC_TUN,GPRHC_TUR,GPRHC_TWN,GPRHC_UKR,GPRHC_USA,GPRHC_VEN,GPRHC_VNM,GPRHC_ZAF,var_name,var_label
0,1900-01-01,NaN,NaN,NaN,87.927849,64.717491,110.453522,NaN,NaN,3.171932,...,0.000000,0.038840,0.000000,0.000000,2.718799,0.051787,0.012947,1.152253,month,Date (year/month)
1,1900-02-01,NaN,NaN,NaN,86.566490,71.936844,96.250488,NaN,NaN,3.122822,...,0.000000,0.125471,0.000000,0.000000,2.732469,0.027882,0.000000,1.143176,GPR,Recent GPR (Index: 1985:2019=100)
2,1900-03-01,NaN,NaN,NaN,72.140701,57.475853,84.499428,NaN,NaN,2.602422,...,0.000000,0.180366,0.000000,0.000000,2.151507,0.025767,0.000000,0.863180,GPRT,Recent GPR Threats (Index: 1985:2019=100)
3,1900-04-01,NaN,NaN,NaN,54.419449,37.326603,65.858208,NaN,NaN,1.963141,...,0.000000,0.066774,0.000000,0.000000,1.776175,0.000000,0.000000,0.641026,GPRA,Recent GPR Acts (Index: 1985:2019=100)
4,1900-05-01,NaN,NaN,NaN,64.405197,48.200008,74.373955,NaN,NaN,2.323370,...,0.000000,0.081522,0.000000,0.000000,1.970109,0.013587,0.000000,0.788043,GPRH,Historical GPR (Index: 1900:2019=100)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509,2025-10-01,154.435577,168.799133,149.261887,132.678070,166.271820,122.520813,4.632116,14896.0,4.786263,...,0.012179,0.292291,0.170503,0.755085,3.215199,0.621118,0.024358,0.085251,NaN,NaN
1510,2025-11-01,104.411270,118.654984,88.926468,90.522110,119.104622,77.962738,3.131695,15870.0,3.265518,...,0.000000,0.194376,0.129584,0.518336,2.164053,0.505378,0.077750,0.090709,NaN,NaN
1511,2025-12-01,130.887665,144.223389,122.132919,112.340988,142.124802,99.599037,3.925824,15207.0,4.052618,...,0.000000,0.159447,0.225884,0.876960,2.870050,0.571353,0.039862,0.039862,NaN,NaN
1512,2026-01-01,167.668213,219.089249,99.685295,128.070435,198.120316,70.965256,5.029014,16027.0,4.620046,...,0.025246,0.302954,0.151477,0.858369,3.395607,1.439031,0.012623,0.012623,NaN,NaN
